# 06 — Full Supervised Baseline

Trains `utils.config.CLASSIFIER_MODEL_NAME` on 100% of available labels —
the upper-bound target every other method is compared against. Uses
`CLASSIFIER_SAMPLE_SIZE` (see Task 11's runtime note) since fine-tuning is
CPU-expensive.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_semisupervised
from utils.modeling import get_predictions, train_model

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

train_sample = stratified_sample(train_clean, config.CLASSIFIER_SAMPLE_SIZE, seed=config.SEED)
print(f"Training on {len(train_sample)} fully-labeled rows (upper bound baseline)")

Training on 152 fully-labeled rows (upper bound baseline)


In [3]:
model, tokenizer = train_model(train_sample, model_name=config.CLASSIFIER_MODEL_NAME, epochs=3)

test_probs = get_predictions(model, tokenizer, test_clean["text"].tolist())
test_preds = test_probs.argmax(axis=1)

results, report, cm = evaluate_semisupervised(
    test_clean["label"].to_numpy(), test_preds, config.CLASS_NAMES,
    save_path=config.RESULTS_DIR / "confusion_matrix_full_supervised.png")
print(report)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_full_supervised.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved full-supervised baseline results.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\ACER\OneDrive\Documents\final-project\.worktrees\autolabel-notebooks\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


              precision    recall  f1-score   support

       World       0.86      0.87      0.87      1900
      Sports       0.96      0.96      0.96      1900
    Business       0.77      0.86      0.81      1900
    Sci/Tech       0.87      0.76      0.81      1900

    accuracy                           0.86      7600
   macro avg       0.86      0.86      0.86      7600
weighted avg       0.86      0.86      0.86      7600

Saved full-supervised baseline results.
